In [1]:
import pandas as pd
import altair as alt
if not hasattr(pd.Series, 'iteritems'):
    pd.Series.iteritems = pd.Series.items
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"
df = pd.read_csv(url, header=None)

In [3]:
df.columns = [
    'datetime', 'city', 'state', 'country', 'shape', 
    'duration_seconds', 'duration_hours_min', 'comments', 
    'date_posted', 'latitude', 'longitude'
]

In [4]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['year'] = df['datetime'].dt.year

In [5]:
df = df.dropna(subset=['datetime', 'shape', 'state'])
df_recent = df[df['year'] >= 1990].copy()

In [6]:
top_shapes = df_recent['shape'].value_counts().head(8).index.tolist()
df_top_shapes = df_recent[df_recent['shape'].isin(top_shapes)]

In [7]:
df_counts = (df_top_shapes.groupby('shape').size()
             .reset_index(name='sightings'))

chart1 = alt.Chart(df_counts).mark_bar().encode(
    x=alt.X('sightings:Q', title='Number of Sightings'),
    y=alt.Y('shape:N', sort='-x', title='UFO Shape'),
    color=alt.Color('shape:N', scale=alt.Scale(scheme='tableau10'), legend=None)
).properties(
    title='Top 8 Most Commonly Reported UFO Shapes (1990-Present)',
    width=500, height=300
)

chart1.save("assets/json/ufo_top_shapes.json")

In [8]:
chart1

alt.Chart(...)

In [9]:
df_yearly = df_recent.groupby(['year', 'shape']).size().reset_index(name='sightings_count')

In [10]:
brush = alt.selection_interval(encodings=['x'])

In [11]:
points = alt.Chart(df_yearly).mark_point(filled=True, size=60).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('sightings_count:Q', title='Number of Sightings'),
    color=alt.Color('shape:N', scale=alt.Scale(scheme='category20'), title='UFO Shape'),
).properties(
    title='UFO Sightings Over Time by Shape',
    width=600,
    height=300
).add_params(
    brush
)

In [12]:
chart2 = points.encode(
    opacity=alt.condition(brush, alt.value(1.0), alt.value(0.1))
)

In [13]:
chart2

alt.Chart(...)

In [14]:
chart2.save("assets/json/ufo_yearly_trend.json")

In [15]:
chart1.to_json("assets/json/ufo_top_shapes.json")
chart2.to_json("assets/json/ufo_yearly_trend.json")

'{\n  "$schema": "https://vega.github.io/schema/vega-lite/v5.20.1.json",\n  "config": {\n    "view": {\n      "continuousHeight": 300,\n      "continuousWidth": 300\n    }\n  },\n  "data": {\n    "name": "data-6fa68442bcfd1e8469f1360cbb51169c"\n  },\n  "datasets": {\n    "data-6fa68442bcfd1e8469f1360cbb51169c": [\n      {\n        "shape": "changing",\n        "sightings_count": 4,\n        "year": 1990\n      },\n      {\n        "shape": "chevron",\n        "sightings_count": 2,\n        "year": 1990\n      },\n      {\n        "shape": "cigar",\n        "sightings_count": 5,\n        "year": 1990\n      },\n      {\n        "shape": "circle",\n        "sightings_count": 12,\n        "year": 1990\n      },\n      {\n        "shape": "cylinder",\n        "sightings_count": 2,\n        "year": 1990\n      },\n      {\n        "shape": "diamond",\n        "sightings_count": 2,\n        "year": 1990\n      },\n      {\n        "shape": "disk",\n        "sightings_count": 27,\n        "ye

I think my local Pandas version has removed .iteritems(), so I added the compatibility patch for Altair to work properly. For the first visualization, I created a horizontal bar chart displaying the top 8 most commonly reported UFO shapes from 1990 to the present day using the standardized UFO dataset. To prepare this data in Python, I loaded the CSV file without a header, assigned descriptive column names(because the original UFO dataset has no column names), converted the raw date strings into proper datetime objects, and filtered the rows to include years from 1990 and  after. I then grouped and counted the occurrences of each shape to isolate the top 8 categories. Regarding design choices, I used nominal encoding for the UFO shapes on the y-axis and quantitative encoding for the sighting counts on the x-axis, sorting the bars in descending order by frequency to make comparisons intuitive. For coloring, I mapped the nominal shape variable to the categorical tableau10 color scheme, which distinctively separates each shape category clearly without introducing any misleading ordinal hierarchy.

The second visualization is an interactive scatter plot illustrating the trend of UFO sightings over time, broken down by shape and aggregated yearly. In the Python analysis notebook, I performed data transformations by grouping the filtered dataset by both the year and the reported shape, computing the total annual sightings count for each category using pandas aggregation functions. For design encodings, I assigned the year to an ordinal x-axis and the sighting count to a quantitative y-axis, while employing the category20 color scheme to map the distinct UFO shapes. To enhance exploration, I integrated an interval selection brush across the timeline axis, which dynamically adjusts the opacity of data points(the unselected points will be greyish) so that unselected years dim to a low transparency while selected time windows remain fully highlighted. This interactivity makes the visualization significantly clearer and more attractive to isolate specific historical periods and trace how individual shape reporting trends evolve over decades.